<a href="https://colab.research.google.com/github/Yash1014-code/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yash1014-code/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [33]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add your Hugging Face token "
        "to Colab Secrets with the name HF_TOKEN."
    )

# Login
login(token=HF_TOKEN, add_to_git_credential=False)

# Create DuckDB connection
con = duckdb.connect()

# Enable Hugging Face Parquet access
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face connection ready!")

Hugging Face connection ready!


In [34]:
months = ["02", "03", "04"]

for month in months:

    source = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month=2026-{month}/*.parquet"
    )

    output = f"/content/{month}_2026.parquet"

    query = f"""
    COPY (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_data_available,
            gsc_impressions,
            gsc_clicks,
            gsc_sum_position,
            ga4_sessions,
            scroll_events
        FROM read_parquet('{source}')
        WHERE gsc_data_available IS TRUE
    )
    TO '{output}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """

    print(f"Downloading month {month}...")
    con.execute(query)
    print(f"Saved: {output}")

print("All three months downloaded successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /content/02_2026.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /content/03_2026.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /content/04_2026.parquet
All three months downloaded successfully!


In [35]:
import os

for month in ["02", "03", "04"]:
    path = f"/content/{month}_2026.parquet"

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{month}_2026.parquet → {size_mb:.2f} MB")
    else:
        print(f"{month}_2026.parquet → NOT FOUND")


02_2026.parquet → 26.40 MB
03_2026.parquet → 39.37 MB
04_2026.parquet → 43.76 MB


In [36]:
import pandas as pd

feb = pd.read_parquet("/content/02_2026.parquet")
mar = pd.read_parquet("/content/03_2026.parquet")
apr = pd.read_parquet("/content/04_2026.parquet")

print("February:", feb.shape)
print("March:", mar.shape)
print("April:", apr.shape)

February: (2621783, 9)
March: (3611061, 9)
April: (3901060, 9)


In [37]:
print("Columns:")
print(feb.columns.tolist())

print("\nFebruary sample:")
display(feb.head())

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'ga4_sessions', 'scroll_events']

February sample:


,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_sessions,scroll_events
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,57,0,1778.0,0.0,0.0
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,13,0,85.0,0.0,0.0
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,59,0,1001.0,2.0,0.0
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,17,0,287.0,0.0,0.0
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,6,0,27.0,0.0,0.0


In [38]:
def aggregate_monthly(df):
    grouped = (
        df.groupby(
            ["client_hash_id", "content_hash_id"],
            as_index=False
        )
        .agg(
            gsc_impressions=("gsc_impressions", "sum"),
            gsc_clicks=("gsc_clicks", "sum"),
            gsc_sum_position=("gsc_sum_position", "sum"),
            ga4_sessions=("ga4_sessions", "sum"),
            scroll_events=("scroll_events", "sum")
        )
    )

    # Impression-weighted average position
    grouped["gsc_avg_position"] = (
        grouped["gsc_sum_position"] /
        grouped["gsc_impressions"].replace(0, pd.NA)
    )

    # CTR
    grouped["ctr"] = (
        grouped["gsc_clicks"] /
        grouped["gsc_impressions"].replace(0, pd.NA)
    )

    return grouped

In [39]:
feb_monthly = aggregate_monthly(feb)
mar_monthly = aggregate_monthly(mar)
apr_monthly = aggregate_monthly(apr)

print("February monthly rows:", len(feb_monthly))
print("March monthly rows:", len(mar_monthly))
print("April monthly rows:", len(apr_monthly))

February monthly rows: 153559
March monthly rows: 176738
April monthly rows: 194760


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and why

I chose **Random Forest** because it can learn relationships between multiple content and performance signals without requiring a simple linear relationship. It also works well for ranking pages based on their expected future performance.

For this lane, the goal is to identify pages that may be good candidates for a **content refresh**. Random Forest can combine signals such as impressions, clicks, position, and engagement-related metrics to produce a useful prediction for prioritization.

I chose this method because it provides a stronger modeling approach than the Week-4 rule-based baseline while remaining practical and interpretable through feature importance. The model is used as **decision support**, not as an automatic decision rule.


In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I used a **time-aware split** rather than a random train-test split. The model uses earlier-period data to predict performance in a later period.

This matches the real decision scenario: information available before the prediction period is used to prioritize pages for the future. A random split could mix past and future observations and give an overly optimistic estimate.

I did not rely on a random client split because the main question is about **future performance over time**, so preserving the time order is more important for this evaluation.


In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I trained a Random Forest model using the same evaluation setup as the baseline. The model was trained using earlier-period data and evaluated on the later-period data.

I compared the model and the Week-4 baseline using the same top-K ranking metric. This makes the comparison fair because both approaches are evaluated on the same pages and future performance window.

In [42]:
# Create training dataset: February -> March
train_data = feb_monthly.merge(
    mar_monthly[
        ["client_hash_id", "content_hash_id", "ctr"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_feb", "_mar")
)

# Create test dataset: March -> April
test_data = mar_monthly.merge(
    apr_monthly[
        ["client_hash_id", "content_hash_id", "ctr"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_mar", "_apr")
)

# Rename future targets
train_data = train_data.rename(columns={"ctr_mar": "future_ctr"})
test_data = test_data.rename(columns={"ctr_apr": "future_ctr"})

# Remove rows where future CTR cannot be calculated
train_data = train_data.dropna(subset=["future_ctr"])
test_data = test_data.dropna(subset=["future_ctr"])

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

Training rows: 134238
Testing rows: 158549


In [43]:
features = [
    "gsc_avg_position",
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "scroll_events"
]

target = "future_ctr"

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Features:")
print(features)

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Features:
['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'scroll_events']

X_train shape: (134238, 5)
X_test shape: (158549, 5)
y_train shape: (134238,)
y_test shape: (158549,)


In [44]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr
import numpy as np

# Make predictions on the test data
model_pred = model.predict(X_test)

# Calculate metrics
model_mae = mean_absolute_error(y_test, model_pred)
model_rmse = np.sqrt(mean_squared_error(y_test, model_pred))
model_spearman = spearmanr(y_test, model_pred).statistic

print(f"MAE: {model_mae:.6f}")
print(f"RMSE: {model_rmse:.6f}")
print(f"Spearman rank correlation: {model_spearman:.6f}")

MAE: 0.004545
RMSE: 0.025830
Spearman rank correlation: 0.174270


In [46]:
# ============================================================
# BUILDING WEEK-4 BASELINE FOR TEST SET
# ============================================================

print("\nBUILDING WEEK-4 BASELINE FOR TEST SET")
print("=" * 80)

baseline_eval = test_data[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_avg_position",
        "gsc_impressions",
        "gsc_clicks",
        "future_ctr"
    ]
].copy()


# ------------------------------------------------------------
# Position tier
# ------------------------------------------------------------

def position_tier_eval(position):

    if pd.isna(position):
        return "unknown"

    elif position <= 3:
        return "top_3"

    elif position <= 10:
        return "page_1"

    elif position <= 20:
        return "striking"

    elif position <= 50:
        return "page_3_5"

    else:
        return "deep"


baseline_eval["position_tier"] = (
    baseline_eval["gsc_avg_position"]
    .apply(position_tier_eval)
)


# ------------------------------------------------------------
# Impression tier
# ------------------------------------------------------------

b_q25, b_q50, b_q75 = (
    baseline_eval["gsc_impressions"]
    .quantile([0.25, 0.50, 0.75])
)


def impression_tier_eval(impressions):

    if pd.isna(impressions):
        return "unknown"

    elif impressions <= b_q25:
        return "low"

    elif impressions <= b_q50:
        return "moderate"

    elif impressions <= b_q75:
        return "good"

    else:
        return "excellent"


baseline_eval["impression_tier"] = (
    baseline_eval["gsc_impressions"]
    .apply(impression_tier_eval)
)


# ------------------------------------------------------------
# March CTR
# ------------------------------------------------------------

baseline_eval["march_ctr"] = np.where(
    baseline_eval["gsc_impressions"] > 0,
    baseline_eval["gsc_clicks"]
    / baseline_eval["gsc_impressions"],
    np.nan
)


# ------------------------------------------------------------
# Position-tier median CTR
# ------------------------------------------------------------

baseline_eval["position_ctr_median"] = (
    baseline_eval
    .groupby("position_tier")["march_ctr"]
    .transform("median")
)


# ------------------------------------------------------------
# CTR gap
# ------------------------------------------------------------

baseline_eval["ctr_gap"] = (
    baseline_eval["position_ctr_median"]
    - baseline_eval["march_ctr"]
).clip(lower=0)


# ------------------------------------------------------------
# Eligibility
# ------------------------------------------------------------

baseline_eval["eligible_for_review"] = (
    baseline_eval["gsc_avg_position"] <= 20
)


# ------------------------------------------------------------
# CTR gap percentile
# ------------------------------------------------------------

baseline_eval["ctr_gap_percentile"] = 0.0

eligible_indices = baseline_eval.index[
    baseline_eval["eligible_for_review"]
]

baseline_eval.loc[
    eligible_indices,
    "ctr_gap_percentile"
] = (
    baseline_eval.loc[
        eligible_indices,
        "ctr_gap"
    ]
    .rank(method="average", pct=True)
)


# ------------------------------------------------------------
# Impression score
# ------------------------------------------------------------

baseline_eval["impression_score"] = (
    baseline_eval["impression_tier"]
    .map({
        "low": 0,
        "moderate": 1,
        "good": 2,
        "excellent": 3,
        "unknown": 0
    })
    .fillna(0)
)


# ------------------------------------------------------------
# Position score
# ------------------------------------------------------------

baseline_eval["position_score"] = (
    baseline_eval["position_tier"]
    .map({
        "top_3": 0,
        "page_1": 1,
        "striking": 1,
        "page_3_5": 0,
        "deep": 0,
        "unknown": 0
    })
    .fillna(0)
)


# ------------------------------------------------------------
# Final baseline score
# ------------------------------------------------------------

baseline_eval["baseline_score"] = np.where(
    baseline_eval["eligible_for_review"],
    (
        baseline_eval["ctr_gap_percentile"] * 6
        + baseline_eval["impression_score"]
        + baseline_eval["position_score"]
    ),
    0
)

print("Baseline created successfully.")
print("Baseline rows:", len(baseline_eval))


BUILDING WEEK-4 BASELINE FOR TEST SET
Baseline created successfully.
Baseline rows: 158549


In [47]:
def top_k_mean_ctr(actual_ctr, score, k):
    top_k_indices = score.nlargest(k).index
    return actual_ctr.loc[top_k_indices].mean()


k_values = [20, 100, 500]

ranking_results = []

for k in k_values:

    model_top_k = top_k_mean_ctr(
        test_data["future_ctr"],
        pd.Series(model_pred, index=test_data.index),
        k
    )

    baseline_top_k = top_k_mean_ctr(
        baseline_eval["future_ctr"],
        baseline_eval["baseline_score"],
        k
    )

    ranking_results.append({
        "k": k,
        "Random Forest top-k mean future CTR": model_top_k,
        "Week-4 baseline top-k mean future CTR": baseline_top_k
    })


ranking_comparison = pd.DataFrame(ranking_results)

print("\nTOP-K RANKING COMPARISON")
print("=" * 80)

print(
    ranking_comparison.to_string(index=False)
)


TOP-K RANKING COMPARISON
  k  Random Forest top-k mean future CTR  Week-4 baseline top-k mean future CTR
 20                             0.065211                               0.000627
100                             0.055425                               0.000731
500                             0.034707                               0.000689


In [48]:
baseline_spearman = spearmanr(
    baseline_eval["future_ctr"],
    baseline_eval["baseline_score"]
).statistic

print(
    "\nWeek-4 baseline Spearman rank correlation:",
    round(baseline_spearman, 6)
)

print(
    "Random Forest Spearman rank correlation:",
    round(model_spearman, 6)
)


Week-4 baseline Spearman rank correlation: 0.169878
Random Forest Spearman rank correlation: 0.17427


In [49]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance = (
    importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nFEATURE IMPORTANCE")
print("=" * 80)
print(importance.to_string(index=False))


FEATURE IMPORTANCE
         feature  importance
gsc_avg_position    0.515397
 gsc_impressions    0.325163
      gsc_clicks    0.156110
    ga4_sessions    0.002836
   scroll_events    0.000495


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest provides a useful additional ranking signal, but its predictions are not exact forecasts. The model performs relatively well for the majority of low-CTR pages but has substantially larger errors for rare high-CTR outcomes. It can also over-predict some pages that later receive little or no CTR. Therefore, the model should be used as a decision-support and prioritization tool rather than as a precise CTR forecast.

In [50]:
# ============================================================
# SECTION 4: ERRORS AND INTERPRETATION
# ============================================================

print("ERRORS AND INTERPRETATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. ADD MODEL PREDICTIONS AND RANK
# ------------------------------------------------------------

test_data["model_prediction"] = model_pred

test_data["model_rank"] = (
    test_data["model_prediction"]
    .rank(method="average", ascending=False)
)

print("Model predictions and ranks added successfully.")


ERRORS AND INTERPRETATION
Model predictions and ranks added successfully.


In [51]:
# ------------------------------------------------------------
# 2. CALCULATE PREDICTION ERRORS
# ------------------------------------------------------------

error_analysis = test_data[
    [
        "client_hash_id",
        "content_hash_id",
        "future_ctr",
        "model_prediction",
        "model_rank"
    ]
].copy()

error_analysis["error"] = (
    error_analysis["model_prediction"]
    - error_analysis["future_ctr"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

print("\nError analysis table created successfully.")
print("Rows:", len(error_analysis))


Error analysis table created successfully.
Rows: 158549


In [52]:
# ------------------------------------------------------------
# 3. OVERALL ERROR SUMMARY
# ------------------------------------------------------------

print("\nOVERALL ERROR SUMMARY")
print("-" * 80)

print(
    "Mean absolute error:",
    round(
        error_analysis["absolute_error"].mean(),
        6
    )
)

print(
    "Median absolute error:",
    round(
        error_analysis["absolute_error"].median(),
        6
    )
)

print(
    "Mean signed error:",
    round(
        error_analysis["error"].mean(),
        6
    )
)

print("""
A positive signed error means the model predicted a higher CTR
than was observed.

A negative signed error means the model predicted a lower CTR
than was observed.
""")


OVERALL ERROR SUMMARY
--------------------------------------------------------------------------------
Mean absolute error: 0.004545
Median absolute error: 0.001631
Mean signed error: 0.000462

A positive signed error means the model predicted a higher CTR
than was observed.

A negative signed error means the model predicted a lower CTR
than was observed.



In [53]:
# ------------------------------------------------------------
# 4. LARGEST UNDER-PREDICTIONS
# ------------------------------------------------------------

largest_under_predictions = (
    error_analysis
    .sort_values("error", ascending=True)
    .head(10)
)

print("\nLARGEST UNDER-PREDICTIONS")
print("-" * 80)

print(
    largest_under_predictions[
        [
            "content_hash_id",
            "future_ctr",
            "model_prediction",
            "error"
        ]
    ].to_string(index=False)
)


LARGEST UNDER-PREDICTIONS
--------------------------------------------------------------------------------
         content_hash_id  future_ctr  model_prediction     error
content_2dd234af5230ae43         1.0          0.000493 -0.999507
content_2d3c83d571ff6295         1.0          0.000726 -0.999274
content_7cd04833a048561c         1.0          0.002354 -0.997646
content_4e4e1e3af7bfbf68         1.0          0.002449 -0.997551
content_ca967915db63de07         1.0          0.002857 -0.997143
content_7dae3c3b40125257         1.0          0.003012 -0.996988
content_e5dd1726c1363d2a         1.0          0.003044 -0.996956
content_d2c8213bf7085978         1.0          0.003165 -0.996835
content_e12bad983c9c9edd         1.0          0.004404 -0.995596
content_409b6617081f924b         1.0          0.004746 -0.995254


In [54]:
# ------------------------------------------------------------
# 5. ERROR BY FUTURE CTR RANGE
# ------------------------------------------------------------

error_analysis["future_ctr_band"] = pd.cut(
    error_analysis["future_ctr"],
    bins=[-0.000001, 0.005, 0.01, 0.03, 0.10, float("inf")],
    labels=[
        "0-0.5%",
        "0.5-1%",
        "1-3%",
        "3-10%",
        "10%+"
    ]
)

error_by_band = (
    error_analysis
    .groupby("future_ctr_band", observed=False)
    .agg(
        rows=("future_ctr", "size"),
        mean_future_ctr=("future_ctr", "mean"),
        mean_absolute_error=("absolute_error", "mean")
    )
    .reset_index()
)

print("\nERROR BY FUTURE CTR RANGE")
print("-" * 80)

print(
    error_by_band.to_string(index=False)
)


ERROR BY FUTURE CTR RANGE
--------------------------------------------------------------------------------
future_ctr_band   rows  mean_future_ctr  mean_absolute_error
         0-0.5% 141920         0.000646             0.002887
         0.5-1%   9913         0.006939             0.003382
           1-3%   5026         0.015523             0.010474
          3-10%   1154         0.051745             0.044121
           10%+    536         0.339289             0.324226


In [55]:
# ------------------------------------------------------------
# 6. TOP MODEL-RANKED PAGES
# ------------------------------------------------------------

top_model_pages = (
    error_analysis
    .sort_values("model_rank")
    .head(20)
)

print("\nTOP 20 MODEL-RANKED PAGES")
print("-" * 80)

print(
    top_model_pages[
        [
            "model_rank",
            "content_hash_id",
            "model_prediction",
            "future_ctr",
            "absolute_error"
        ]
    ].to_string(index=False)
)


TOP 20 MODEL-RANKED PAGES
--------------------------------------------------------------------------------
 model_rank          content_hash_id  model_prediction  future_ctr  absolute_error
        1.0 content_fc665b5b67f70a49          0.136186    0.000000        0.136186
        2.0 content_51134f7afb2272b3          0.133101    0.000000        0.133101
        3.0 content_5d7b110b50cae5c8          0.123544    0.142857        0.019313
        4.0 content_8ca396e3bf7dd1df          0.113338    0.000000        0.113338
        5.0 content_4190ffb9428c90ff          0.113165    0.000000        0.113165
        6.0 content_564a2a55f60aeac6          0.109953    0.012725        0.097228
        7.5 content_f0ace4b0a4654b9f          0.109393    0.000000        0.109393
        7.5 content_b15af0d801e3888a          0.109393    0.142857        0.033464
        9.0 content_d5520f1aa4b1d630          0.109264    0.035714        0.073549
       10.0 content_08c2642f9bc04c04          0.108996    0.06

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.